In [ ]:
import os
import numpy as np
from tqdm import tqdm

subject = 1
SESSIONS = range(1, 40 + 1)

target_dir = os.path.join(
    "/media/harveylab/STORAGE1_NA/NSD/full_brain_max", f"subj{subject:02d}"
)

# Sicherstellen, dass das Zielverzeichnis existiert
os.makedirs(target_dir, exist_ok=True)

all_betas = []  # Liste zur Speicherung aller geladenen Beta-Werte

for session in tqdm(SESSIONS, desc="Loading Sessions"):
    target_file_path = os.path.join(
        target_dir, f"normalized_betas_session_{session}.npy"
    )

    # Betas aus der Datei laden und zur Liste hinzufügen
    if os.path.exists(target_file_path):
        betas = np.load(target_file_path)
        all_betas.append(betas)
    else:
        print(f"Warnung: Datei {target_file_path} nicht gefunden. Überspringe...")

# Alle Sessions entlang der ersten Achse (Samples) konkatenieren
if all_betas:
    concatenated_betas = np.concatenate(all_betas, axis=0)
    
    # Datei speichern
    concatenated_file_path = os.path.join(target_dir, "normalized_betas_all_sessions.npy")
    np.save(concatenated_file_path, concatenated_betas)
    print(f"Gespeichert: {concatenated_file_path}")
else:
    print("Keine Betas gefunden. Überprüfe die Quelldateien.")

In [ ]:
import os
import numpy as np
import psutil  # Für RAM/CPU-Überwachung
from tqdm import tqdm

def get_memory_usage():
    """Gibt die aktuelle RAM-Auslastung zurück."""
    process = psutil.Process(os.getpid())  # Eigenes Python-Skript überwachen
    mem_info = psutil.virtual_memory()  # Gesamter RAM
    return {
        "RAM genutzt (MB)": process.memory_info().rss / 1e6,
        "RAM verfügbar (MB)": mem_info.available / 1e6,
        "RAM gesamt (MB)": mem_info.total / 1e6,
        "RAM genutzt (%)": mem_info.percent,
    }

subject = 1
SESSIONS = range(1, 40 + 1)

target_dir = os.path.join(
    "/media/harveylab/STORAGE1_NA/NSD/full_brain_max", f"subj{subject:02d}"
)

# Sicherstellen, dass das Zielverzeichnis existiert
os.makedirs(target_dir, exist_ok=True)

all_betas = []  # Liste zur Speicherung aller geladenen Beta-Werte

for session in tqdm(SESSIONS, desc="Loading Sessions"):
    target_file_path = os.path.join(
        target_dir, f"normalized_betas_session_{session}.npy"
    )

    # RAM-Nutzung vor dem Laden ausgeben
    print(f"Session {session}: RAM-Status vor Laden:", get_memory_usage())

    # Betas aus der Datei laden und zur Liste hinzufügen
    if os.path.exists(target_file_path):
        betas = np.load(target_file_path)
        all_betas.append(betas)
    else:
        print(f"Warnung: Datei {target_file_path} nicht gefunden. Überspringe...")

    # RAM-Nutzung nach dem Laden ausgeben
    print(f"Session {session}: RAM-Status nach Laden:", get_memory_usage())

# Alle Sessions entlang der ersten Achse (Samples) konkatenieren
if all_betas:
    print("RAM-Status vor Concatenation:", get_memory_usage())
    
    concatenated_betas = np.concatenate(all_betas, axis=0)

    print("RAM-Status nach Concatenation:", get_memory_usage())

    # Datei speichern
    concatenated_file_path = os.path.join(target_dir, "normalized_betas_all_sessions.npy")
    np.save(concatenated_file_path, concatenated_betas)
    
    print(f"Gespeichert: {concatenated_file_path}")
    print("Finaler RAM-Status:", get_memory_usage())
else:
    print("Keine Betas gefunden. Überprüfe die Quelldateien.")


In [ ]:
import os
import numpy as np
import psutil
from tqdm import tqdm

def get_memory_usage():
    """Gibt die aktuelle RAM-Auslastung zurück."""
    process = psutil.Process(os.getpid())  
    mem_info = psutil.virtual_memory()
    return {
        "RAM genutzt (MB)": process.memory_info().rss / 1e6,
        "RAM verfügbar (MB)": mem_info.available / 1e6,
        "RAM gesamt (MB)": mem_info.total / 1e6,
        "RAM genutzt (%)": mem_info.percent,
    }

subject = 1
SESSIONS = range(1, 40 + 1)

target_dir = os.path.join(
    "/media/harveylab/STORAGE1_NA/NSD/full_brain_max", f"subj{subject:02d}"
)
os.makedirs(target_dir, exist_ok=True)

concatenated_file_path = os.path.join(target_dir, "normalized_betas_all_sessions.npy")

# Erste Datei laden, um die Dimensionen zu bestimmen
first_session_file = os.path.join(target_dir, f"normalized_betas_session_1.npy")
if not os.path.exists(first_session_file):
    raise FileNotFoundError(f"Fehlende Datei: {first_session_file}")

first_betas = np.load(first_session_file)
n_samples, n_voxels = first_betas.shape  # Dimensionen pro Session bestimmen

# Memmap-Datei auf Festplatte anlegen (Initialisierung)
concatenated_betas = np.memmap(concatenated_file_path, dtype=np.float32, mode='w+', shape=(n_samples * len(SESSIONS), n_voxels))

# Inkrementell Daten laden & schreiben (statt alles in RAM zu speichern)
start_idx = 0
for session in tqdm(SESSIONS, desc="Processing Sessions"):
    session_file_path = os.path.join(target_dir, f"normalized_betas_session_{session}.npy")
    
    if os.path.exists(session_file_path):
        print(f"Lade Session {session}... RAM-Status:", get_memory_usage())
        
        betas = np.load(session_file_path)
        end_idx = start_idx + betas.shape[0]

        # Direkt in die `np.memmap` Datei schreiben
        concatenated_betas[start_idx:end_idx] = betas
        start_idx = end_idx  # Update Start-Index für nächste Session
    else:
        print(f"Warnung: {session_file_path} nicht gefunden. Überspringe...")

# Sicherstellen, dass die Daten geschrieben werden
concatenated_betas.flush()

print(f"Gespeicherte Datei: {concatenated_file_path}")
print("Finaler RAM-Status:", get_memory_usage())
